# WFS Diffraction: does it bias the danish wavefront fit?

**Author:** Aaron Roodman   **Created:** 2026-06-17   **Status:** in progress

**Motivation.** danish fits donuts in the **geometric (photon-ray) approximation**, ignoring
diffraction. Three RubinTV data observations (WFS and FAM) point to a real intra/extra
asymmetry the geometric model can't make: (1) the spherical term **Z11 differs ~0.3 um** between
**separately**-fit intra and extra donuts; (2) **rim residuals**, especially intra; (3) struts
sharper extra. This notebook tests whether **diffraction** drives it, on-axis, by generating
diffraction donuts and fitting them with danish, both **paired** and **unpaired**.

**Donut generator.** A defocused Rubin donut is huge (~13 arcsec), so an out-of-focus FFT PSF
needs a large pupil grid; galsim's `OpticalPSF` auto-grid undersamples it (square artifact) and
batoid's `huygensPSF` (Zemax-style Huygens) refocuses / needs dx~lambda/D/20 (overnight). The
reliable tool here is a **manual Fraunhofer FFT** on an explicit N=2048 annular pupil:
`field = aperture * exp(2*pi*i*W/lambda)`, `donut = |FFT(field)|^2`, with `W` the off-axis
annular-Zernike wavefront from the ts_wep `Instrument` (`getOffAxisCoeff`). It is validated below
against the geometric ray-traced donut (same size, clean hole). Struts are omitted (only
single-blade radial struts are easy to model; LSST's double-bladed spider is not).

**Key result (on-axis, FAM +/-1.5 mm, i-band, design spherical Z11~0.29 waves):** the geometric
donut fits with ~0 intra/extra Z11 split; the **diffraction donut gives a +0.11 um Z11 split in
the separate (unpaired) fits** -- same sign as the data, ~1/3 the magnitude -- and that split is
**averaged away in the paired fit**, exactly matching the observation that only separate fits
expose it. The remaining gap to ~0.3 um is plausibly the pure Fresnel near-field term (Huygens),
larger spherical at the actual WFS field positions, and the 8 mm giant-donut defocus.

## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-06-17 | Aaron Roodman | Initial: validated manual-Fraunhofer-FFT diffraction donut (N=2048 annular pupil, off-axis annular-Zernike wavefront, no struts), production danish fit config (21-term Noll 4-19/22-26, fwhm bound [0.5,1.5], startWithIntrinsic, x_scale=jac, unbinned), geometric-vs-FFT intra/extra fit (paired + unpaired). Finding: diffraction gives a +0.11 um Z11 intra/extra split in the unpaired fits (right sign, ~1/3 of the observed 0.3 um), hidden in the paired fit. |

## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Helper Functions](#functions)
4. [Generate & Validate the Donuts](#donuts)
5. [Intra vs Extra: spherical breaks the symmetry](#intraextra)
6. [Danish Fits: Z11 intra/extra split (paired & unpaired)](#fits)
7. [Data / Model / Residual](#resid)
8. [Summary](#summary)

<a id='params'></a>
## 1. Parameters

In [ ]:
# ============================================================
# Parameters
# ============================================================
detector      = "R30_S21"     # science det -> FAM (camera-hexapod) instrument; FoV-centre fit (theta=0)
band          = "i"
wl_nm         = 754.0         # i-band effective wavelength (nm)
defocal_mm    = 1.5           # FAM camera-hexapod defocus
seeing_arcsec = 0.7           # Gaussian seeing FWHM
npix          = 185           # detector stamp (pixels), 0.2"/pix
peak_e        = 1.0e4         # donut peak electrons/pixel
sky_e         = 200.0         # sky pedestal (e/pix) -> finite background
# Production danish fit config (FAM-style, FoV centre):
noll          = [4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 22, 23, 24, 25, 26]  # standard 21
fwhm_bounds   = (0.5, 1.5)    # arcsec; contains the danish x0 (1.0), caps the runaway
# Fraunhofer-FFT pupil grid:
Ngrid         = 2048          # pupil-plane grid (compare 4096 if memory allows)
gridpad       = 1.15          # pupil half-width = gridpad * R_outer
n_band        = 9             # broadband wavelength samples across i-band
cmap          = "inferno"

<a id='setup'></a>
## 2. Setup & Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import batoid
import danish
import galsim.zernike as gz
from scipy.ndimage import gaussian_filter, zoom
from lsst.ts.wep.utils import getTaskInstrument, DefocalType, BandLabel
from lsst.ts.wep.image import Image
from lsst.ts.wep.estimation import WfEstimator

inst = getTaskInstrument("LSSTCam", detector)
inst.defocalOffset = defocal_mm * 1e-3
fid = inst.getBatoidModel(band)
wl = wl_nm * 1e-9
R = inst.radius; Rin = R * inst.obscuration
pix = inst.pixelSize; fl = inst.focalLength
detplate = np.degrees(pix / fl) * 3600          # detector arcsec/pixel (0.2)
print(f"R_outer={R:.3f} m, obscuration={inst.obscuration:.3f}, detector {detplate:.3f} arcsec/pix")
print(f"on-axis design Z11 (spherical) = "
      f"{1e6*inst.getOffAxisCoeff(0,0,DefocalType.Extra,'r',nollIndicesModel=np.arange(79),nollIndicesIntr=np.arange(4,23))[11]:.3f} um "
      f"({inst.getOffAxisCoeff(0,0,DefocalType.Extra,'r',nollIndicesModel=np.arange(79),nollIndicesIntr=np.arange(4,23))[11]/wl:.2f} waves)")

<a id='functions'></a>
## 3. Helper Functions

In [ ]:
# ---- pupil grid (built once) ----
_u = np.linspace(-R * gridpad, R * gridpad, Ngrid)
_U, _V = np.meshgrid(_u, _u)
_rho = np.hypot(_U, _V)
_du = _u[1] - _u[0]
aperture = (_rho >= Rin) & (_rho <= R)          # annulus; NO struts


def fft_donut(dtype, wl_):
    """Monochromatic Fraunhofer diffraction donut on the N=2048 pupil grid, using the off-axis
    annular-Zernike wavefront for this defocal side. Returns (donut_2048, arcsec/FFT-pixel)."""
    ab = inst.getOffAxisCoeff(0, 0, dtype, band, nollIndicesModel=np.arange(79), nollIndicesIntr=np.arange(4, 23))
    W = np.where(aperture, gz.Zernike(ab, R_outer=R, R_inner=Rin)(_U, _V), 0.0)   # OPD, metres
    field = aperture * np.exp(2j * np.pi * W / wl_)
    donut = np.abs(np.fft.fftshift(np.fft.fft2(np.fft.ifftshift(field)))) ** 2
    return donut, np.degrees(wl_ / (Ngrid * _du)) * 3600


def to_detector(donut, fft_plate):
    """Seeing-convolve at the FINE FFT scale, then downsample to the detector grid and
    centre in an npix stamp. (Seeing before downsample, else the sharp rings alias.)"""
    s = gaussian_filter(np.clip(donut, 0, None), (seeing_arcsec / 2.355) / fft_plate)
    z = zoom(s, fft_plate / detplate)
    n = z.shape[0]; out = np.zeros((npix, npix))
    if n <= npix:                                # donut smaller than the stamp -> pad/centre
        o = (npix - n) // 2; out[o:o + n, o:o + n] = z
    else:                                        # larger -> centre-crop
        c = n // 2; h = npix // 2; out = z[c - h:c - h + npix, c - h:c - h + npix]
    return out


def diffraction_donut(dtype, seed, broadband=True):
    """Detector-grid diffraction donut + Poisson noise. Broadband sums over the i-band so the
    monochromatic Fresnel rings wash out, as in real data."""
    if broadband:
        wls = np.linspace(0.692, 0.818, n_band) * 1e-6
        wts = np.exp(-0.5 * ((wls - wl) / 0.05e-6) ** 2); wts /= wts.sum()
    else:
        wls, wts = [wl], [1.0]
    acc = np.zeros((npix, npix))
    for wl_, wt in zip(wls, wts):
        d, pl = fft_donut(dtype, wl_); acc += wt * to_detector(d, pl)
    rng = np.random.default_rng(seed)
    return rng.poisson(acc * peak_e / acc.max() + sky_e).astype(float)


def geom_donut(dtype, seed, osamp=4):
    """Geometric (photon-ray) donut: 2-D histogram of un-vignetted ray landings (no diffraction)."""
    sign = +1 if dtype == DefocalType.Extra else -1
    tel = fid.withLocallyShiftedOptic("LSSTCamera", [0, 0, sign * defocal_mm * 1e-3])
    r = batoid.RayVector.asPolar(optic=tel, wavelength=wl, theta_x=0, theta_y=0, nrad=300, naz=1800, inner=0)
    tel.trace(r); ok = ~r.vignetted & ~r.failed
    x, y = r.x[ok], r.y[ok]; cx, cy = x.mean(), y.mean()
    half = npix * pix / 2; nb = npix * osamp
    H, _, _ = np.histogram2d(x, y, bins=nb, range=[[cx - half, cx + half], [cy - half, cy + half]])
    H = gaussian_filter(H.T, (seeing_arcsec / 2.355) * np.radians(1 / 3600) * fl / (pix / osamp))
    img = H.reshape(npix, osamp, npix, osamp).sum(axis=(1, 3))
    rng = np.random.default_rng(seed)
    return rng.poisson(img * peak_e / img.max() + sky_e).astype(float)


# ---- danish fit (production config) ----
_w0 = WfEstimator(algoName="danish", algoConfig={"jointFitPair": False}, instConfig=inst, nollIndices=np.array(noll))
_zr = inst.getOffAxisCoeff(0, 0, DefocalType.Extra, band, nollIndicesModel=np.arange(79), nollIndicesIntr=np.arange(4, 23))
_fac = danish.DonutFactory(R_outer=R, R_inner=Rin, mask_params=inst.maskParams, focal_length=fl, pixel_scale=pix)
_m = danish.SingleDonutModel(_fac, z_ref=_zr, z_terms=tuple(noll), thx=0, thy=0, npix=npix, bkg_order=_w0.algo.bkgOrder)
_b = _m.pack_params(flux=[-np.inf, np.inf], dx=[-np.inf, np.inf], dy=[-np.inf, np.inf],
                    fwhm=list(fwhm_bounds), bkg=[[-np.inf, np.inf]] * _m.nbkg, z_fit=[[-np.inf, np.inf]] * len(noll))
SINGLE_BOUNDS = tuple([list(x) for x in zip(*_b)])     # fwhm bounded [0.5,1.5], rest free


def single_fit(img, dtype):
    """Unpaired (single-sided) danish fit; returns (zk_um, fwhm, history)."""
    wfe = WfEstimator(algoName="danish",
                      algoConfig={"jointFitPair": False, "binning": 1, "lstsqKwargs": {"bounds": SINGLE_BOUNDS, "x_scale": "jac"}},
                      instConfig=inst, nollIndices=np.array(noll), startWithIntrinsic=True, units="um", saveHistory=True)
    zk, meta = wfe.estimateZk(Image(img, (0.0, 0.0), dtype, BandLabel.LSST_I), None)
    return np.asarray(zk).ravel(), float(meta.get("fwhm", np.nan)), wfe.history


def paired_fit(imgE, imgI):
    """Paired (joint) danish fit; one shared wavefront + fwhm."""
    wfe = WfEstimator(algoName="danish", algoConfig={"jointFitPair": True, "binning": 1, "lstsqKwargs": {"x_scale": "jac"}},
                      instConfig=inst, nollIndices=np.array(noll), startWithIntrinsic=True, units="um", saveHistory=True)
    zk, meta = wfe.estimateZk(Image(imgE, (0.0, 0.0), DefocalType.Extra, BandLabel.LSST_I),
                              Image(imgI, (0.0, 0.0), DefocalType.Intra, BandLabel.LSST_I))
    return np.asarray(zk).ravel(), float(meta.get("fwhm", np.nan)), wfe.history


i11 = noll.index(11)   # index of Z11 (spherical) in the fitted vector

<a id='donuts'></a>
## 4. Generate & Validate the Donuts

Geometric (ray) and Fraunhofer-FFT (diffraction) donuts, extra and intra, on the detector grid. The FFT donut should match the geometric one in size and have a clean central hole; the difference is the diffraction ringing at the rim.

In [ ]:
donuts = {
    "geometric": {"extra": geom_donut(DefocalType.Extra, 1), "intra": geom_donut(DefocalType.Intra, 2)},
    "FFT diffraction": {"extra": diffraction_donut(DefocalType.Extra, 1), "intra": diffraction_donut(DefocalType.Intra, 2)},
}
fig, axes = plt.subplots(2, 2, figsize=(8, 8), constrained_layout=True)
for col, kind in enumerate(["geometric", "FFT diffraction"]):
    for row, side in enumerate(["extra", "intra"]):
        axes[row, col].imshow(donuts[kind][side], origin="lower", cmap=cmap)
        axes[row, col].set_title(f"{kind} - {side}", fontsize=10); axes[row, col].set_xticks([]); axes[row, col].set_yticks([])
fig.suptitle(f"On-axis donuts (FAM +/-{defocal_mm} mm, {band}-band, {seeing_arcsec} arcsec seeing)", fontsize=12)
plt.show()

yy, xx = np.mgrid[0:npix, 0:npix]; c = (npix - 1) / 2; rr = np.hypot(xx - c, yy - c)
for kind in ["geometric", "FFT diffraction"]:
    a = donuts[kind]["extra"] - sky_e; a = a / a.max()
    print(f"{kind:16s}: central(r<25)={a[rr<25].mean():.3f}  outer half-max r={rr[a>0.5].max():.0f} px")

<a id='intraextra'></a>
## 5. Intra vs Extra: spherical breaks the symmetry

For *pure* defocus the Fraunhofer intra and extra donuts are identical (conjugate symmetry). With the design spherical aberration present, the defocus*spherical interaction makes them differ - and the difference is concentrated at the inner/outer **rim**, matching the data's rim residuals. (Full-resolution FFT donuts, no seeing, to show the structure.)

In [ ]:
E2, plE = fft_donut(DefocalType.Extra, wl); I2, plI = fft_donut(DefocalType.Intra, wl)
E2 = E2 / E2.max(); I2 = I2 / I2.max()
cc = Ngrid // 2; ww = 850
yy2, xx2 = np.mgrid[0:Ngrid, 0:Ngrid]; rr2 = np.hypot(xx2 - cc, yy2 - cc)
nb = 90; e = np.linspace(0, 16, nb + 1) / plE; rax = 0.5 * (e[:-1] + e[1:]) * plE
pE = np.array([E2[(rr2 >= e[k]) & (rr2 < e[k + 1])].mean() for k in range(nb)])
pI = np.array([I2[(rr2 >= e[k]) & (rr2 < e[k + 1])].mean() for k in range(nb)])
print(f"extra-intra donut RMS diff = {np.sqrt(np.mean((E2-I2)**2)):.4f} (peak-normed)")
fig, ax = plt.subplots(1, 2, figsize=(13, 5.2), constrained_layout=True)
ext = ww * plE
dimg = (E2 - I2)[cc - ww:cc + ww, cc - ww:cc + ww]; v = np.percentile(np.abs(dimg), 99.5)
im = ax[0].imshow(dimg, origin="lower", cmap="RdBu_r", vmin=-v, vmax=v, extent=[-ext, ext, -ext, ext])
plt.colorbar(im, ax=ax[0], fraction=0.046); ax[0].set_title("EXTRA - INTRA (rim difference)"); ax[0].set_xlabel("arcsec")
ax[1].plot(rax, pE / np.nanmax(pE), label="extra"); ax[1].plot(rax, pI / np.nanmax(pI), label="intra")
ax[1].set_xlabel("radius [arcsec]"); ax[1].set_ylabel("norm azimuthal mean"); ax[1].legend(); ax[1].grid(alpha=0.3)
ax[1].set_title("intra vs extra radial profile")
fig.suptitle("FFT donut intra vs extra (on-axis): spherical breaks the symmetry at the rim", fontsize=12)
plt.show()

<a id='fits'></a>
## 6. Danish Fits: Z11 intra/extra split (paired & unpaired)

Fit each donut with the production danish config. A **paired** fit forces one wavefront for the pair (averaging any intra/extra disagreement); **unpaired** fits expose it. We compare the spherical term **Z11** between intra and extra for the geometric (diffraction-free) and FFT (diffraction) donuts.

In [ ]:
results = {}
print(f"{'donut':<16s}{'Z11 extra':>11s}{'Z11 intra':>11s}{'Z11 I-E (unpaired)':>20s}{'Z11 (paired)':>14s}{'fwhm':>8s}{'resid':>8s}")
for kind in ["geometric", "FFT diffraction"]:
    zE, fE, hE = single_fit(donuts[kind]["extra"], DefocalType.Extra)
    zI, fI, hI = single_fit(donuts[kind]["intra"], DefocalType.Intra)
    zp, fp, hp = paired_fit(donuts[kind]["extra"], donuts[kind]["intra"])
    rE = np.std(hE["extra"]["image"] - hE["extra"]["model"])
    results[kind] = dict(zE=zE, zI=zI, fE=fE, fI=fI, zp=zp, fp=fp, hE=hE, hI=hI)
    print(f"{kind:<16s}{zE[i11]:>11.3f}{zI[i11]:>11.3f}{zI[i11]-zE[i11]:>20.3f}{zp[i11]:>14.3f}{fE:>8.2f}{rE:>8.0f}")
print()
print("Diffraction induces a Z11 intra/extra split in the UNPAIRED fits that is absent geometrically,")
print("and is averaged away in the PAIRED fit -- matching the data (split only seen in separate fits).")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4.6), constrained_layout=True)
kinds = ["geometric", "FFT diffraction"]; x = np.arange(len(kinds))
a1.bar(x, [1e3 * (results[k]["zI"][i11] - results[k]["zE"][i11]) for k in kinds], color=["C0", "C3"])
a1.axhline(300, color="k", ls=":", lw=1); a1.text(0, 300, " ~data (0.3 um)", fontsize=8, va="bottom")
a1.set_xticks(x); a1.set_xticklabels(kinds); a1.set_ylabel("Z11 intra - extra [nm]")
a1.set_title("Unpaired-fit spherical (Z11) intra/extra split"); a1.grid(alpha=0.3, axis="y")
# full Zernike intra-extra for the FFT donut
d = results["FFT diffraction"]["zI"] - results["FFT diffraction"]["zE"]
a2.bar(range(len(noll)), 1e3 * d, color="C3"); a2.axvline(i11, color="k", ls=":", lw=1)
a2.set_xticks(range(len(noll))); a2.set_xticklabels([f"Z{j}" for j in noll], fontsize=7, rotation=90)
a2.set_ylabel("intra - extra [nm]"); a2.set_title("FFT donut: intra-extra by Zernike (Z11 marked)"); a2.grid(alpha=0.3, axis="y")
plt.show()

<a id='resid'></a>
## 7. Data / Model / Residual

Unpaired data / model / residual for the FFT (diffraction) donuts. The residual carries the diffraction structure the geometric model cannot represent (rim ringing), and it differs between intra and extra.

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(11, 7.4), constrained_layout=True)
for row, side in enumerate(["extra", "intra"]):
    h = results["FFT diffraction"]["hE" if side == "extra" else "hI"]
    data = h[side]["image"]; model = h[side]["model"]; resid = data - model
    fw = results["FFT diffraction"]["fE" if side == "extra" else "fI"]
    for ax, arr, ttl, cm in [(axs[row, 0], data, "data", cmap), (axs[row, 1], model, "model", cmap),
                             (axs[row, 2], resid, "residual", "RdBu_r")]:
        if ttl == "residual":
            v = np.nanpercentile(np.abs(arr), 99.5); ax.imshow(arr, origin="lower", cmap=cm, vmin=-v, vmax=v)
        else:
            ax.imshow(arr, origin="lower", cmap=cm)
        ax.set_title(f"FFT {side} (fwhm {fw:.2f}) - {ttl}", fontsize=9); ax.set_xticks([]); ax.set_yticks([])
plt.show()

<a id='summary'></a>
## 8. Summary

On-axis, FAM +/-1.5 mm, i-band, design spherical Z11 ~ 0.29 waves, production danish config
(21-term Noll 4-19/22-26, fwhm bound [0.5,1.5], startWithIntrinsic, x_scale='jac', unbinned):

| donut | Z11 intra-extra (unpaired) | Z11 (paired) | fwhm |
|---|---|---|---|
| geometric (no diffraction) | ~ -0.01 um | +0.06 um | 0.53 |
| **FFT diffraction** | **+0.11 um** | +0.02 um | 0.71 |
| **data (RubinTV)** | **~0.3 um** | - | - |

**Conclusions.**
- **Diffraction biases the separately-fit spherical term by ~+0.11 um** (right sign, ~1/3 of the
  observed ~0.3 um), where the geometric donut shows ~0. danish's geometric model cannot represent
  the diffraction rim structure, and absorbs it into Z11 (and a larger fwhm, 0.71 vs 0.53).
- The split is **only visible in the unpaired fits** -- the paired fit averages it away
  (+0.02 um) -- exactly as observed: the data Z11 disagreement shows up when intra and extra are
  fit separately.

**Caveats / next steps.** This is the **Fraunhofer (far-field)** part on-axis; the pure
**Fresnel near-field** term (Zemax-style Huygens, dx~lambda/D/20, an overnight run) would add more.
The full ~0.3 um likely also needs the **larger spherical at the actual WFS field positions**
(off-axis) and the **8 mm giant-donut defocus**. The validated FFT generator here (N=2048 annular
pupil; compare 4096 for convergence) is the basis for those follow-ups. Note the FFT donut is
intra/extra-symmetric only through spherical (a far-field limitation); a genuinely different
intra vs extra donut requires the Fresnel/Huygens treatment.